# Notebook 06 — Gate Controller
**Mission Veteran AI by G3 · Veteran Career Navigator**  

Tests `evaluate_export_gate` — scans a tagged resume draft for INFERRED and
MISSING sentences, uses PT-004 to generate plain-English hold explanations,
and returns gate status OPEN or BLOCKED.

**Gate logic:**  
- Gate is **OPEN** when every sentence is EVIDENCED and the grounding check
  passed (i.e., `grounding_downgrade_count == 0`).  
- Gate is **BLOCKED** when any INFERRED or MISSING sentence exists, **or** when
  the grounding check downgraded one or more EVIDENCED items (`grounding_downgrade_count > 0`).  
- Grounding-check holds surface first in the holds list, before tag-based holds.  
- Export is only permitted when gate is OPEN.

**Prerequisites**
- Notebook 05 completed (tagged resume draft available)  
- `.env` with `WATSONX_API_KEY`, `WATSONX_URL`, `WATSONX_PROJECT_ID`

In [ ]:
%pip install -q \
    ibm-watsonx-ai \
    python-dotenv \
    ibm-aigov-facts-client>=2.0.0

In [ ]:
import json, os, re, sys
from dotenv import load_dotenv

load_dotenv()
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

API_KEY    = os.environ['WATSONX_API_KEY']
WX_URL     = os.environ.get('WATSONX_URL', 'https://us-south.ml.cloud.ibm.com')
PROJECT_ID = os.environ['WATSONX_PROJECT_ID']
print('Credentials loaded ✓')

## 1 · Load PT-004 from tblPrompts

In [ ]:
KB_PATH = os.path.join(ROOT, 'knowledge-bases', 'prompt_templates.json')
with open(KB_PATH, 'r', encoding='utf-8') as f:
    templates = json.load(f)

template = next(t for t in templates if t['name'] == 'gate_hold_explanation')
print(f"Loaded: {template['template_id']} — {template['description']}")

## 2 · Sample tagged resume draft (from Notebook 05 or direct input)

In [ ]:
# Simulated output from build_resume_draft — matches current toolkit output schema
# build_resume_draft returns resume_sections (dict of section → list of tagged items),
# evidenced_count, inferred_count, missing_count, grounding_warnings, disclaimer.
RESUME_DRAFT = {
    'resume_sections': {
        'summary': [
            {'text': 'Results-driven supply chain professional with 4+ years of military logistics experience, proven team leadership, and a track record of measurable operational improvements.', 'tag': 'EVIDENCED'},
        ],
        'experience': [
            {'text': 'Directed a 12-member logistics team, improving on-time delivery from 82% to 96% over 18 months.', 'tag': 'EVIDENCED'},
            {'text': 'Managed a $2.1M annual inventory with a 98.7% accountability rate.', 'tag': 'EVIDENCED'},
            {'text': 'Applied lean process improvement methodologies to reduce supply chain waste.', 'tag': 'INFERRED'},
            {'text': 'Managed ERP system configuration and vendor portal administration.', 'tag': 'MISSING'},
            {'text': 'Coordinated cross-functional teams across three regional distribution centers.', 'tag': 'INFERRED'},
        ],
        'skills': [
            {'text': 'Logistics Coordination', 'tag': 'EVIDENCED'},
            {'text': 'Inventory Management', 'tag': 'EVIDENCED'},
            {'text': 'Team Leadership', 'tag': 'EVIDENCED'},
            {'text': 'Budget Oversight', 'tag': 'EVIDENCED'},
            {'text': 'Cross-functional Collaboration', 'tag': 'EVIDENCED'},
        ],
        'education_certifications': [],
    },
    'evidenced_count': 8,
    'inferred_count': 2,
    'missing_count': 1,
    'grounding_warnings': [
        {'section': 'experience', 'text': 'Applied lean process improvement methodologies to reduce supply chain waste.', 'reason': 'Lean methodology not mentioned in service summary; downgraded from EVIDENCED.'},
    ],
    'disclaimer': 'DRAFT – Review before use.',
}
print('Sample draft loaded ✓')

## 3 · Collect all blocked sentences

In [ ]:
blocked = []
sections = RESUME_DRAFT.get('resume_sections', {})
for section_items in sections.values():
    for item in (section_items if isinstance(section_items, list) else [section_items]):
        if isinstance(item, dict) and item.get('tag', '').upper() in ('INFERRED', 'MISSING'):
            blocked.append({'sentence': item.get('text',''), 'tag': item['tag'].upper()})

# Also surface grounding warnings
grounding_warnings = RESUME_DRAFT.get('grounding_warnings', [])

print(f'Blocked sentences: {len(blocked)}')
for b in blocked:
    print(f"  [{b['tag']}] {b['sentence']}")
if grounding_warnings:
    print(f'\nGrounding warnings: {len(grounding_warnings)}')
    for w in grounding_warnings:
        print(f"  ⚠  [{w.get('section','')}] {w.get('text','')}")
        if w.get('reason'):
            print(f"      reason: {w.get('reason','')}")

## 4 · Call Granite with PT-004 for plain-English hold explanations

In [ ]:
from ibm_watsonx_ai import APIClient, Credentials
from ibm_watsonx_ai.foundation_models import ModelInference
from ibm_watsonx_ai.metanames import GenTextParamsMetaNames as Params

MODEL_ID = 'ibm/granite-4-h-small'
p        = template['model_params']

model = ModelInference(
    model_id=MODEL_ID,
    api_client=APIClient(credentials=Credentials(url=WX_URL, api_key=API_KEY)),
    project_id=PROJECT_ID,
    params={
        Params.MAX_NEW_TOKENS:    p['max_new_tokens'],
        Params.MIN_NEW_TOKENS:    p['min_new_tokens'],
        Params.TEMPERATURE:       p['temperature'],
        Params.REPETITION_PENALTY:p['repetition_penalty'],
    },
)

blocked_text = json.dumps(
    [{'sentence': b['sentence'], 'tag': b['tag']} for b in blocked],
    indent=2
)
user_prompt = template['user_prompt_template'].format(blocked_sentences=blocked_text)

response = model.generate(
    prompt=f"System:\n{template['system_prompt']}\nUser:\n{user_prompt}\nAssistant:\n"
)

raw = response['results'][0]['generated_text'].strip()
raw = re.sub(r'^```[a-z]*\n?', '', raw)
raw = re.sub(r'\n?```$', '', raw)
print('Raw hold explanations:', raw[:300], '...')

## 5 · Display gate hold report

In [ ]:
holds = json.loads(raw)

print('═' * 70)
print('EXPORT GATE: BLOCKED')
print('═' * 70)

for i, hold in enumerate(holds, 1):
    tag            = hold.get('tag', '?')
    sentence       = hold.get('sentence', '')
    explanation    = hold.get('explanation', '')
    action         = hold.get('action_required', '')
    icon           = '⚠' if tag == 'INFERRED' else '✗'
    print(f'\nHold #{i} [{tag}] {icon}')
    print(f'  Sentence:   {sentence}')
    print(f'  Reason:     {explanation}')
    print(f'  Action:     {action}')

print()
print(f'Total holds: {len(holds)}')
print('Resolve all holds before exporting.')

## 6 · Simulate resolving a hold and re-evaluating the gate

In [ ]:
# Simulate user confirming the INFERRED sentences and removing the MISSING sentence
revised_sections = {
    'summary': RESUME_DRAFT['resume_sections']['summary'],
    'experience': [
        {'text': 'Directed a 12-member logistics team, improving on-time delivery from 82% to 96% over 18 months.', 'tag': 'EVIDENCED'},
        {'text': 'Managed a $2.1M annual inventory with a 98.7% accountability rate.', 'tag': 'EVIDENCED'},
        # Confirmed INFERRED → EVIDENCED by user
        {'text': 'Applied lean process improvement methodologies to reduce supply chain waste.', 'tag': 'EVIDENCED'},
        # MISSING sentence removed by user
        # Confirmed INFERRED → EVIDENCED by user
        {'text': 'Coordinated cross-functional teams across three regional distribution centers.', 'tag': 'EVIDENCED'},
    ],
    'skills': RESUME_DRAFT['resume_sections']['skills'],
    'education_certifications': [],
}
revised_draft = {**RESUME_DRAFT, 'resume_sections': revised_sections, 'grounding_warnings': []}

# Re-evaluate using the same section-iterating logic
still_blocked = [
    item
    for section_items in revised_draft['resume_sections'].values()
    for item in (section_items if isinstance(section_items, list) else [section_items])
    if isinstance(item, dict) and item.get('tag', '').upper() in ('INFERRED', 'MISSING')
]

gate_status = 'OPEN' if not still_blocked and not revised_draft['grounding_warnings'] else 'BLOCKED'
print(f'Revised gate status: {gate_status}')
if gate_status == 'OPEN':
    print('✓ All sentences evidenced. Document is ready to export.')